# Model v1 (simplified) - CatBoost baseline for VIEWS forecasting


## 0. Setup

**What we do:** import libraries and define paths.  
**Outcome:** stable execution across machines (paths can be overridden via env vars).

In [58]:
import os
import json
from datetime import timedelta

import numpy as np
import pandas as pd

from catboost import CatBoostRegressor, Pool
import os
import sys

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

notebook_dir = os.path.dirname(os.path.abspath("__file__"))
sys.path.append('..')
base_dir = os.path.abspath(os.path.join("..", ".."))
print(base_dir)

# Paths (override if needed)
ALLDATA_PATH = os.path.join(base_dir, "data", "AllData.csv")
TESTDATA_PATH = os.path.join(base_dir, "data", "TestDataset.csv")  # optional
ARTIFACTS_DIR = os.getenv("ARTIFACTS_DIR", "artifacts")
OUTPUTS_DIR = os.getenv("OUTPUTS_DIR", "outputs")
HOLDOUT_DAYS = int(os.getenv("HOLDOUT_DAYS", "30"))
MODEL_TAG = os.getenv("MODEL_TAG", "v1_simplified")

# Feature/model knobs
DEDUPLICATE = os.getenv("DEDUPLICATE", "1") == "1"
USE_CHANNEL_ID = os.getenv("USE_CHANNEL_ID", "0") == "1"
LOSS_FUNCTION = os.getenv("LOSS_FUNCTION", "MAE")
EVAL_METRIC = os.getenv("EVAL_METRIC", LOSS_FUNCTION)
ALPHA_CHANNEL = float(os.getenv("ALPHA_CHANNEL", "30"))
ALPHA_SLOPE = float(os.getenv("ALPHA_SLOPE", "100"))
MIN_SLOPE_ROWS = int(os.getenv("MIN_SLOPE_ROWS", "20"))
USE_PRED_CLIP = os.getenv("USE_PRED_CLIP", "1") == "1"
PRED_CLIP_Q = float(os.getenv("PRED_CLIP_Q", "0.65"))
MIN_CLIP_ROWS = int(os.getenv("MIN_CLIP_ROWS", "50"))
BLEND_ALPHA = float(os.getenv("BLEND_ALPHA", "0.3"))
BLEND_BASE = os.getenv("BLEND_BASE", "ch_med")
SCALE_FACTOR = float(os.getenv("SCALE_FACTOR", "0.9"))

# EVAL_METRIC="MAE"
# PRED_CLIP_Q=0.60
# BLEND_ALPHA=0.30
# BLEND_BASE="ch_med"
# SCALE_FACTOR=0.90




os.makedirs(ARTIFACTS_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

print("ALLDATA_PATH:", ALLDATA_PATH)
print("TESTDATA_PATH:", TESTDATA_PATH)
print("ARTIFACTS_DIR:", ARTIFACTS_DIR)
print("OUTPUTS_DIR:", OUTPUTS_DIR)
print("HOLDOUT_DAYS:", HOLDOUT_DAYS)
print("MODEL_TAG:", MODEL_TAG)
print("DEDUPLICATE:", DEDUPLICATE)
print("USE_CHANNEL_ID:", USE_CHANNEL_ID)
print("LOSS_FUNCTION:", LOSS_FUNCTION)
print("EVAL_METRIC:", EVAL_METRIC)
print("ALPHA_CHANNEL:", ALPHA_CHANNEL)
print("ALPHA_SLOPE:", ALPHA_SLOPE)
print("MIN_SLOPE_ROWS:", MIN_SLOPE_ROWS)
print("USE_PRED_CLIP:", USE_PRED_CLIP)
print("PRED_CLIP_Q:", PRED_CLIP_Q)
print("MIN_CLIP_ROWS:", MIN_CLIP_ROWS)
print("BLEND_ALPHA:", BLEND_ALPHA)
print("BLEND_BASE:", BLEND_BASE)
print("SCALE_FACTOR:", SCALE_FACTOR)


/Users/karimkhabib/Documents/Projects Programming/PyCharm/telegram-ads-forecaster
ALLDATA_PATH: /Users/karimkhabib/Documents/Projects Programming/PyCharm/telegram-ads-forecaster/data/AllData.csv
TESTDATA_PATH: /Users/karimkhabib/Documents/Projects Programming/PyCharm/telegram-ads-forecaster/data/TestDataset.csv
ARTIFACTS_DIR: artifacts
OUTPUTS_DIR: outputs
HOLDOUT_DAYS: 30
MODEL_TAG: v1_simplified
DEDUPLICATE: True
USE_CHANNEL_ID: False
LOSS_FUNCTION: MAE
EVAL_METRIC: MAE
ALPHA_CHANNEL: 30.0
ALPHA_SLOPE: 100.0
MIN_SLOPE_ROWS: 20
USE_PRED_CLIP: True
PRED_CLIP_Q: 0.65
MIN_CLIP_ROWS: 50
BLEND_ALPHA: 0.3
BLEND_BASE: ch_med
SCALE_FACTOR: 0.9


In [59]:
test_df = pd.read_csv(TESTDATA_PATH)
test_df.head()

,CPM,CHANNEL_NAME,DATE,VIEWS
0,25.0,topcareerschool,2023-03-11,NaN
1,10.0,topcareerschool,2023-03-11,NaN
2,13.0,hrsecrets_life,2023-03-11,NaN
3,7.0,hrsecrets_life,2023-03-11,NaN
4,7.0,Dirclub,2023-03-12,NaN


## 1. Load data and diagnostics

**What we do:** read `AllData.csv`, normalize column names, parse `DATE`, check data quality.  
**Outcome:** a clean `df` with diagnostics to identify potential issues.

In [60]:
df = pd.read_csv(ALLDATA_PATH)

# Dataset quirk: sometimes columns have leading/trailing spaces
df.columns = df.columns.str.strip()

print(f"Initial shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

# Parse DATE
df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce")

required = {"CPM", "CHANNEL_NAME", "DATE", "VIEWS"}
missing = required - set(df.columns)
assert not missing, f"Missing required columns: {missing}"

# Data quality checks
print("=== Data Quality Checks ===")
print(f"Total rows: {len(df):,}")
print(f"Missing DATE: {df['DATE'].isna().sum()}")
print(f"Missing VIEWS: {df['VIEWS'].isna().sum()}")
print(f"Missing CPM: {df['CPM'].isna().sum()}")
print(f"Missing CHANNEL_NAME: {df['CHANNEL_NAME'].isna().sum()}")

# Remove rows with missing critical data
initial_len = len(df)
df = df.dropna(subset=["DATE", "VIEWS", "CPM", "CHANNEL_NAME"]).copy()
print(f"Rows after cleaning: {len(df):,} (removed {initial_len - len(df)})")

# Optional: deduplicate identical inputs to reduce label noise
if DEDUPLICATE:
    key = ["CHANNEL_NAME", "DATE", "CPM"]
    df = (
        df.groupby(key, as_index=False)
          .agg(
              VIEWS=("VIEWS", "median"),
              CLICKS=("CLICKS", "sum"),
              ACTIONS=("ACTIONS", "sum"),
              dup_count=("VIEWS", "size"),
          )
    )
    df["dup_count"] = df["dup_count"].astype(int)
    print(f"After dedup: {len(df):,} rows")

# VIEWS distribution diagnostics
print("=== VIEWS Distribution ===")
print(df["VIEWS"].describe())
print(f"VIEWS == 0: {(df['VIEWS'] == 0).sum()} ({(df['VIEWS'] == 0).mean()*100:.2f}%)")
print(f"VIEWS < 0: {(df['VIEWS'] < 0).sum()}")
print(f"VIEWS > 100000: {(df['VIEWS'] > 100000).sum()} ({(df['VIEWS'] > 100000).mean()*100:.2f}%)")

# CPM distribution diagnostics
print("=== CPM Distribution ===")
print(df["CPM"].describe())
print(f"CPM <= 0: {(df['CPM'] <= 0).sum()}")
print(f"CPM > 100: {(df['CPM'] > 100).sum()} ({(df['CPM'] > 100).mean()*100:.2f}%)")

# Date range
print("=== Date Range ===")
print(f"Min date: {df['DATE'].min()}")
print(f"Max date: {df['DATE'].max()}")
print(f"Days span: {(df['DATE'].max() - df['DATE'].min()).days}")
print(f"Unique dates: {df['DATE'].nunique()}")

# Channel statistics
print("=== Channel Statistics ===")
print(f"Unique channels: {df['CHANNEL_NAME'].nunique()}")
print(f"Channels with < 10 samples: {(df['CHANNEL_NAME'].value_counts() < 10).sum()}")
print(f"Top 10 channels by frequency:")
print(df['CHANNEL_NAME'].value_counts().head(10))

# Check for potential issues
print("=== Potential Issues ===")
if (df['VIEWS'] == 0).mean() > 0.5:
    print("WARNING: More than 50% of VIEWS are zero - model may struggle")
if (df['VIEWS'] < 0).any():
    print("WARNING: Negative VIEWS found - will clip to 0")
if (df['CPM'] <= 0).any():
    print("WARNING: Non-positive CPM found")
if df['CHANNEL_NAME'].isna().any():
    print("WARNING: Missing channel names found")

df.head()


Initial shape: (142609, 7)
Columns: ['AD_ID', 'CPM', 'VIEWS', 'CLICKS', 'ACTIONS', 'CHANNEL_NAME', 'DATE']
=== Data Quality Checks ===
Total rows: 142,609
Missing DATE: 0
Missing VIEWS: 0
Missing CPM: 0
Missing CHANNEL_NAME: 0
Rows after cleaning: 142,609 (removed 0)
After dedup: 126,841 rows
=== VIEWS Distribution ===
count    1.268410e+05
mean     1.030361e+03
std      7.466483e+03
min      0.000000e+00
25%      6.650000e+01
50%      2.600000e+02
75%      6.690000e+02
max      1.136470e+06
Name: VIEWS, dtype: float64
VIEWS == 0: 38 (0.03%)
VIEWS < 0: 0
VIEWS > 100000: 72 (0.06%)
=== CPM Distribution ===
count    126841.000000
mean          9.132017
std          30.599184
min           1.000000
25%           2.000000
50%           3.610000
75%           8.400000
max         999.910000
Name: CPM, dtype: float64
CPM <= 0: 0
CPM > 100: 693 (0.55%)
=== Date Range ===
Min date: 2024-10-02 00:00:00
Max date: 2025-12-17 00:00:00
Days span: 441
Unique dates: 430
=== Channel Statistics ===
Uni

,CHANNEL_NAME,DATE,CPM,VIEWS,CLICKS,ACTIONS,dup_count
0,A117_SportMotion,2024-11-04,3.6,3890.0,135,16,1
1,A117_SportMotion,2025-02-06,2.0,2815.0,64,6,1
2,A117_SportMotion,2025-09-05,2.0,1075.0,27,5,1
3,A117_SportMotion,2025-10-24,5.0,1985.0,112,18,1
4,A1BOUTIQUE,2025-03-01,2.2,13.0,0,0,1


## 2. Time-based split (holdout last N days)

**What we do:** split by time for honest evaluation.  
**Why:** random split leaks time patterns.  
**Outcome:** `train_df` and `valid_df` for local scoring.

In [61]:
def split_last_days(data: pd.DataFrame, holdout_days: int = 30):
    unique_dates = np.sort(data["DATE"].unique())
    if len(unique_dates) <= holdout_days:
        raise ValueError(
            f"Not enough unique dates ({len(unique_dates)}) for holdout_days={holdout_days}"
        )
    cutoff = unique_dates[-holdout_days]
    train = data.loc[data["DATE"] < cutoff].copy()
    valid = data.loc[data["DATE"] >= cutoff].copy()
    return train, valid, cutoff, unique_dates[-1]

train_df, valid_df, cutoff, dmax = split_last_days(df, holdout_days=HOLDOUT_DAYS)

print("Train:", train_df.shape, "Valid:", valid_df.shape)
print("Cutoff:", cutoff, "→", dmax)


Train: (117950, 7) Valid: (8891, 7)
Cutoff: 2025-11-18T00:00:00.000000000 → 2025-12-17T00:00:00.000000000


## 3. Feature engineering (minimal)

We build a compact feature set focused on stable signal and interpretability.

**CPM features:**
- `log_cpm` - log1p(CPM) to handle heavy tails
- `cpm_to_ch_median` - CPM relative to the channel median

**Channel features (aggregated):**
- `ch_med_smooth` - shrinked median of log1p(VIEWS) per channel
- `ch_count_log` - log1p(number of rows per channel)
- `ch_ctr_smooth` - shrinked clicks/views per channel
- `ch_actions_rate_smooth` - shrinked actions/views per channel
- `ch_slope_smooth`, `ch_log_pred` - channel elasticity to CPM (shrinked)
- `CHANNEL_NAME` (optional categorical, controlled by `USE_CHANNEL_ID`)

**Date features (yearless seasonality):**
- `dow`, `is_weekend`
- `month`
- `doy_sin`, `doy_cos`

**Post-processing (optional):**
- prediction clipping by channel/global quantiles
- optional blending with `ch_med` or `ch_log_pred`
- optional scale factor


In [62]:
# Use yearless seasonality features (safe for 2023 dates in test)

def add_date_features(data: pd.DataFrame) -> pd.DataFrame:
    out = data.copy()
    d = out["DATE"]

    out["dow"] = d.dt.dayofweek.astype(int)
    out["is_weekend"] = (out["dow"] >= 5).astype(int)

    out["month"] = d.dt.month.astype(int)
    out["dayofyear"] = d.dt.dayofyear.astype(int)

    # Cyclic encoding for day-of-year
    out["doy_sin"] = np.sin(2 * np.pi * out["dayofyear"] / 366.0)
    out["doy_cos"] = np.cos(2 * np.pi * out["dayofyear"] / 366.0)

    return out


def fit_preprocess_params(train: pd.DataFrame) -> dict:
    # Reserved for future use (kept for consistency)
    return {}


def apply_basic_preprocess(data: pd.DataFrame, params=None) -> pd.DataFrame:
    out = data.copy()

    # CPM features
    out["cpm"] = out["CPM"].astype(float)
    out["log_cpm"] = np.log1p(out["cpm"].clip(lower=0))

    # DATE features
    out = add_date_features(out)

    return out


def _safe_slope(x, y):
    if len(x) < 2 or np.var(x) == 0:
        return np.nan
    return float(np.polyfit(x, y, 1)[0])


def fit_channel_stats(train: pd.DataFrame, alpha: float = 10.0):
    # Robust channel stats on train only (log scale), with shrinkage.
    y = np.log1p(train["VIEWS"].clip(lower=0))
    x = np.log1p(train["CPM"].clip(lower=0))
    global_med = float(np.median(y))

    global_ctr = float(train["CLICKS"].sum() / train["VIEWS"].sum())
    global_actions_rate = float(train["ACTIONS"].sum() / train["VIEWS"].sum())

    global_slope = _safe_slope(x, y)
    global_cpm_log_med = float(np.median(x))
    global_clip = float(train["VIEWS"].quantile(PRED_CLIP_Q))

    stats = (
        train.assign(y=y, log_cpm=x)
             .groupby("CHANNEL_NAME")
             .agg(
                 ch_count=("VIEWS", "size"),
                 ch_med=("y", "median"),
                 views_sum=("VIEWS", "sum"),
                 clicks_sum=("CLICKS", "sum"),
                 actions_sum=("ACTIONS", "sum"),
                 ch_cpm_log_med=("log_cpm", "median"),
             )
             .reset_index()
    )

    # Channel CPM slope (log-log), shrinked to global slope
    def _channel_slope(g):
        if len(g) < MIN_SLOPE_ROWS or g["CPM"].nunique() < 2:
            return np.nan
        gx = np.log1p(g["CPM"].clip(lower=0))
        gy = np.log1p(g["VIEWS"].clip(lower=0))
        return _safe_slope(gx, gy)

    ch_slope = (
        train.groupby("CHANNEL_NAME")
             .apply(_channel_slope)
             .rename("ch_slope_raw")
             .reset_index()
    )

    stats = stats.merge(ch_slope, on="CHANNEL_NAME", how="left")

    stats["ch_ctr_raw"] = stats["clicks_sum"] / stats["views_sum"].replace(0, np.nan)
    stats["ch_actions_rate_raw"] = stats["actions_sum"] / stats["views_sum"].replace(0, np.nan)

    w = stats["ch_count"] / (stats["ch_count"] + alpha)
    stats["ch_med_smooth"] = w * stats["ch_med"] + (1 - w) * global_med
    stats["ch_ctr_smooth"] = w * stats["ch_ctr_raw"].fillna(global_ctr) + (1 - w) * global_ctr
    stats["ch_actions_rate_smooth"] = (
        w * stats["ch_actions_rate_raw"].fillna(global_actions_rate)
        + (1 - w) * global_actions_rate
    )

    # Slope shrinkage uses a separate alpha
    w_slope = stats["ch_count"] / (stats["ch_count"] + ALPHA_SLOPE)
    stats["ch_slope_smooth"] = (
        w_slope * stats["ch_slope_raw"].fillna(global_slope)
        + (1 - w_slope) * global_slope
    )

    # Per-channel prediction clip
    ch_clip = (
        train.groupby("CHANNEL_NAME")["VIEWS"]
             .quantile(PRED_CLIP_Q)
             .rename("ch_views_clip")
             .reset_index()
    )
    stats = stats.merge(ch_clip, on="CHANNEL_NAME", how="left")
    stats.loc[stats["ch_count"] < MIN_CLIP_ROWS, "ch_views_clip"] = np.nan

    keep_cols = [
        "CHANNEL_NAME",
        "ch_count",
        "ch_med_smooth",
        "ch_ctr_smooth",
        "ch_actions_rate_smooth",
        "ch_cpm_log_med",
        "ch_slope_smooth",
        "ch_views_clip",
    ]

    return (
        stats[keep_cols],
        global_med,
        global_ctr,
        global_actions_rate,
        global_slope,
        global_cpm_log_med,
        global_clip,
    )


def apply_channel_stats(
    data: pd.DataFrame,
    ch_stats: pd.DataFrame,
    global_med: float,
    global_ctr: float,
    global_actions_rate: float,
    global_slope: float,
    global_cpm_log_med: float,
    global_clip: float,
) -> pd.DataFrame:
    out = data.merge(ch_stats, on="CHANNEL_NAME", how="left")
    out["ch_count"] = out["ch_count"].fillna(0).astype(int)
    out["ch_count_log"] = np.log1p(out["ch_count"])
    out["ch_med_smooth"] = out["ch_med_smooth"].fillna(global_med).astype(float)
    out["ch_ctr_smooth"] = out["ch_ctr_smooth"].fillna(global_ctr).astype(float)
    out["ch_actions_rate_smooth"] = out["ch_actions_rate_smooth"].fillna(global_actions_rate).astype(float)
    out["ch_slope_smooth"] = out["ch_slope_smooth"].fillna(global_slope).astype(float)
    out["ch_cpm_log_med"] = out["ch_cpm_log_med"].fillna(global_cpm_log_med).astype(float)
    out["ch_views_clip"] = out["ch_views_clip"].fillna(global_clip).astype(float)

    # Expected log view given channel slope and CPM deviation
    out["ch_log_pred"] = out["ch_med_smooth"] + out["ch_slope_smooth"] * (out["log_cpm"] - out["ch_cpm_log_med"])

    return out


def fit_channel_cpm_stats(train: pd.DataFrame):
    stats = (
        train.groupby("CHANNEL_NAME")["CPM"]
             .median()
             .reset_index(name="ch_cpm_median")
    )
    global_median = float(train["CPM"].median())
    return stats, global_median


def apply_channel_cpm_stats(data: pd.DataFrame, ch_stats: pd.DataFrame, global_median: float) -> pd.DataFrame:
    out = data.merge(ch_stats, on="CHANNEL_NAME", how="left")
    out["ch_cpm_median"] = out["ch_cpm_median"].fillna(global_median)
    denom = out["ch_cpm_median"].replace(0, np.nan)
    out["cpm_to_ch_median"] = (out["cpm"] / denom).fillna(1.0)
    return out


def post_process_predictions(pred, feats, global_clip, use_clip=True, blend_alpha=-1.0, blend_base="ch_med", scale_factor=1.0):
    out = np.clip(pred, 0, None)
    if scale_factor != 1.0:
        out = out * scale_factor
    if blend_alpha >= 0:
        if blend_base == "ch_log_pred":
            base = np.expm1(feats["ch_log_pred"].to_numpy())
        else:
            base = np.expm1(feats["ch_med_smooth"].to_numpy())
        out = blend_alpha * out + (1 - blend_alpha) * base
    if use_clip:
        clip_val = feats["ch_views_clip"].to_numpy()
        out = np.minimum(out, clip_val)
    return out


## 4. Build train/valid matrices

**What we do:** fit preprocessing on train only, build features, create CatBoost Pools.  
**Outcome:** `train_pool`, `valid_pool` and feature specification.

In [63]:
# Fit preprocess on train only (no leakage)
preprocess = fit_preprocess_params(train_df)

# Basic features
train_fe = apply_basic_preprocess(train_df, preprocess)
valid_fe = apply_basic_preprocess(valid_df, preprocess)

# Channel stats (train only)
(
    ch_stats,
    global_med_log,
    global_ctr,
    global_actions_rate,
    global_slope,
    global_cpm_log_med,
    global_views_clip,
) = fit_channel_stats(train_df, alpha=ALPHA_CHANNEL)

train_fe = apply_channel_stats(
    train_fe,
    ch_stats,
    global_med_log,
    global_ctr,
    global_actions_rate,
    global_slope,
    global_cpm_log_med,
    global_views_clip,
)
valid_fe = apply_channel_stats(
    valid_fe,
    ch_stats,
    global_med_log,
    global_ctr,
    global_actions_rate,
    global_slope,
    global_cpm_log_med,
    global_views_clip,
)

# Channel CPM stats (train only)
ch_cpm_stats, global_cpm_median = fit_channel_cpm_stats(train_df)
train_fe = apply_channel_cpm_stats(train_fe, ch_cpm_stats, global_cpm_median)
valid_fe = apply_channel_cpm_stats(valid_fe, ch_cpm_stats, global_cpm_median)

# Feature spec (minimal and stable)
feature_cols = [
    "log_cpm",
    "cpm_to_ch_median",
    "dow", "is_weekend", "month", "doy_sin", "doy_cos",
    "ch_count_log", "ch_med_smooth", "ch_ctr_smooth", "ch_actions_rate_smooth",
    "ch_slope_smooth", "ch_log_pred",
]

cat_features = []
if USE_CHANNEL_ID:
    feature_cols.append("CHANNEL_NAME")
    cat_features = ["CHANNEL_NAME"]

X_train = train_fe[feature_cols]
X_valid = valid_fe[feature_cols]

y_train_raw = train_df["VIEWS"].astype(float).values
y_valid_raw = valid_df["VIEWS"].astype(float).values

y_train = np.log1p(np.clip(y_train_raw, 0, None))
y_valid = np.log1p(np.clip(y_valid_raw, 0, None))

train_weights = train_df["dup_count"].values if "dup_count" in train_df.columns else None
valid_weights = valid_df["dup_count"].values if "dup_count" in valid_df.columns else None

train_pool = Pool(X_train, y_train, cat_features=cat_features, weight=train_weights)
valid_pool = Pool(X_valid, y_valid, cat_features=cat_features, weight=valid_weights)

X_train.head()


,log_cpm,cpm_to_ch_median,dow,is_weekend,month,doy_sin,doy_cos,ch_count_log,ch_med_smooth,ch_ctr_smooth,ch_actions_rate_smooth,ch_slope_smooth,ch_log_pred
0,1.526056,1.285714,0,0,11,-0.829677,0.558244,1.609438,5.853722,0.016213,0.004384,-0.39939,5.768364
1,1.098612,0.714286,3,0,2,0.593327,0.804962,1.609438,5.853722,0.016213,0.004384,-0.39939,5.939081
2,1.098612,0.714286,4,0,9,-0.898292,-0.439400,1.609438,5.853722,0.016213,0.004384,-0.39939,5.939081
3,1.791759,1.785714,4,0,10,-0.926324,0.376728,1.609438,5.853722,0.016213,0.004384,-0.39939,5.662245
4,1.163151,0.550000,5,1,3,0.857315,0.514793,2.197225,5.232019,0.014898,0.003912,-0.39939,5.402110


## 5. Train CatBoost (v1 simplified)


In [64]:
# Helper function for metrics calculation
def calculate_metrics(y_true, y_pred, name=""):
    # Calculate comprehensive metrics.
    y_true = np.asarray(y_true, dtype=float).clip(min=0)
    y_pred = np.asarray(y_pred, dtype=float).clip(min=0)

    mae = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

    # RMSLE
    rmsle = float(np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2)))

    # SMAPE
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    denom = np.where(denom == 0, 1, denom)
    smape = float(np.mean(np.abs(y_pred - y_true) / denom)) * 100

    metrics = {
        "MAE": mae,
        "RMSE": rmse,
        "RMSLE": rmsle,
        "SMAPE": smape
    }

    if name:
        print(f"{name} Metrics:")
        for k, v in metrics.items():
            print(f"  {k}: {v:.4f}")

    return metrics

print("=" * 70)
print(f"Training CatBoost on log1p(VIEWS) | loss={LOSS_FUNCTION}")
print("=" * 70)

model = CatBoostRegressor(
    loss_function=LOSS_FUNCTION,
    depth=8,
    learning_rate=0.05,
    iterations=4000,
    l2_leaf_reg=5,
    random_strength=1.0,
    bootstrap_type="Bayesian",
    bagging_temperature=0.8,
    random_seed=RANDOM_SEED,
    eval_metric=EVAL_METRIC,
    verbose=500,
    od_type="Iter",
    od_wait=200,
    task_type="CPU",
    devices="0",
)

model.fit(train_pool, eval_set=valid_pool, use_best_model=True, verbose=500)

# Predictions on validation (convert back to raw scale)
pred_valid_log = model.predict(valid_pool)
pred_valid_raw = np.clip(np.expm1(pred_valid_log), 0, None)

pred_valid = post_process_predictions(
    pred_valid_raw,
    valid_fe,
    global_views_clip,
    use_clip=USE_PRED_CLIP,
    blend_alpha=BLEND_ALPHA,
    blend_base=BLEND_BASE,
    scale_factor=SCALE_FACTOR,
)

metrics_valid = calculate_metrics(y_valid_raw, pred_valid, "CatBoost (log1p target)")


Training CatBoost on log1p(VIEWS) | loss=MAE
0:	learn: 1.4336375	test: 1.6243694	best: 1.6243694 (0)	total: 14.9ms	remaining: 59.6s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 1.463556711
bestIteration = 39

Shrink model to first 40 iterations.
CatBoost (log1p target) Metrics:
  MAE: 656.7197
  RMSE: 6026.2547
  RMSLE: 2.0352
  SMAPE: 105.3601


## 6. Evaluate (raw scale)

We predict in log space and invert back with `expm1`.  
Outcome: local metrics you can compare to baselines and track between versions.

In [65]:
def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_true - y_pred)))

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def rmsle(y_true, y_pred):
    y_true = np.clip(np.asarray(y_true, dtype=float), 0, None)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0, None)
    return float(np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2)))

def smape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.abs(y_true) + np.abs(y_pred) + eps
    return float(np.mean(2.0 * np.abs(y_pred - y_true) / denom))

metrics_local = {
    "MAE": mae(y_valid_raw, pred_valid),
    "RMSE": rmse(y_valid_raw, pred_valid),
    "RMSLE": rmsle(y_valid_raw, pred_valid),
    "SMAPE": smape(y_valid_raw, pred_valid),
}
metrics_local


{'MAE': 656.7196946145312,
 'RMSE': 6026.254714272827,
 'RMSLE': 2.035181269182409,
 'SMAPE': 1.0536005843583016}

## 7. Baseline comparison (same split)

Outcome: sanity check. The simplified model should beat these baselines.


In [66]:
global_median = float(train_df["VIEWS"].median())

# Baseline 1: global median
b1 = np.full_like(y_valid_raw, fill_value=global_median, dtype=float)

# Baseline 2: DOW median
train_dow = train_df["DATE"].dt.dayofweek
valid_dow = valid_df["DATE"].dt.dayofweek
dow_median = train_df.groupby(train_dow)["VIEWS"].median()
b2 = valid_dow.map(dow_median).fillna(global_median).to_numpy(dtype=float)

# Baseline 3: channel shrink baseline (raw scale)
alpha = 5
ch_raw = (train_df.groupby("CHANNEL_NAME")["VIEWS"].agg(ch_median="median", ch_count="size").reset_index())
valid_ch = valid_df[["CHANNEL_NAME"]].merge(ch_raw, on="CHANNEL_NAME", how="left")
w = (valid_ch["ch_count"] / (valid_ch["ch_count"] + alpha)).fillna(0.0)
b3 = (w * valid_ch["ch_median"].fillna(global_median) + (1 - w) * global_median).to_numpy(dtype=float)

def score_row(name, pred):
    return {
        "model": name,
        "MAE": mae(y_valid_raw, pred),
        "RMSE": rmse(y_valid_raw, pred),
        "RMSLE": rmsle(y_valid_raw, pred),
        "SMAPE": smape(y_valid_raw, pred),
    }

rows = [
    score_row("baseline_global_median", b1),
    score_row("baseline_dow_median", b2),
    score_row("baseline_channel_shrink", b3),
    score_row("catboost_v1_simplified", pred_valid),
]
pd.DataFrame(rows).sort_values("MAE")


,model,MAE,RMSE,RMSLE,SMAPE
3,catboost_v1_simplified,656.719695,6026.254714,2.035181,1.053601
0,baseline_global_median,685.683444,6038.175369,2.157317,1.105368
2,baseline_channel_shrink,687.178766,5968.714879,2.071089,1.055325
1,baseline_dow_median,688.481779,6039.450677,2.170283,1.107201


## 8. Final training on FULL data (for submission)

**Important:** once you selected a configuration, train on **all** rows to avoid losing the last N days.  
Outcome: `final_model` + full artifacts for submission and deployment.

In [67]:
full_df = df.copy()

# Fit preprocessing on FULL data
preprocess_full = fit_preprocess_params(full_df)
full_fe = apply_basic_preprocess(full_df, preprocess_full)

# Fit channel stats on FULL data (still offline, no leakage for submission)
(
    ch_stats_full,
    global_med_log_full,
    global_ctr_full,
    global_actions_rate_full,
    global_slope_full,
    global_cpm_log_med_full,
    global_views_clip_full,
) = fit_channel_stats(full_df, alpha=ALPHA_CHANNEL)

full_fe = apply_channel_stats(
    full_fe,
    ch_stats_full,
    global_med_log_full,
    global_ctr_full,
    global_actions_rate_full,
    global_slope_full,
    global_cpm_log_med_full,
    global_views_clip_full,
)

# Fit channel CPM stats on FULL data
ch_cpm_stats_full, global_cpm_median_full = fit_channel_cpm_stats(full_df)
full_fe = apply_channel_cpm_stats(full_fe, ch_cpm_stats_full, global_cpm_median_full)

X_full = full_fe[feature_cols]
y_full_raw = full_df["VIEWS"].astype(float).values
y_full_log = np.log1p(np.clip(y_full_raw, 0, None))

full_weights = full_df["dup_count"].values if "dup_count" in full_df.columns else None
full_pool = Pool(X_full, y_full_log, cat_features=cat_features, weight=full_weights)

final_model = CatBoostRegressor(**model.get_params())
final_model.fit(full_pool, verbose=200)


0:	learn: 1.4453517	total: 12.6ms	remaining: 50.3s
200:	learn: 0.7557640	total: 2.6s	remaining: 49.2s
400:	learn: 0.7101121	total: 4.75s	remaining: 42.6s
600:	learn: 0.6802737	total: 6.36s	remaining: 36s
800:	learn: 0.6604119	total: 8.06s	remaining: 32.2s
1000:	learn: 0.6440567	total: 9.69s	remaining: 29s
1200:	learn: 0.6313297	total: 11.4s	remaining: 26.6s
1400:	learn: 0.6212570	total: 12.9s	remaining: 23.9s
1600:	learn: 0.6122623	total: 14.3s	remaining: 21.5s
1800:	learn: 0.6040291	total: 15.8s	remaining: 19.3s
2000:	learn: 0.5967795	total: 17.3s	remaining: 17.3s
2200:	learn: 0.5897346	total: 18.9s	remaining: 15.4s
2400:	learn: 0.5835816	total: 20.3s	remaining: 13.5s
2600:	learn: 0.5781546	total: 21.8s	remaining: 11.7s
2800:	learn: 0.5733548	total: 23.3s	remaining: 9.96s
3000:	learn: 0.5687409	total: 24.8s	remaining: 8.26s
3200:	learn: 0.5646807	total: 26.3s	remaining: 6.56s
3400:	learn: 0.5610162	total: 27.7s	remaining: 4.88s
3600:	learn: 0.5574564	total: 29.2s	remaining: 3.23s
3800

## 9. Save artifacts

Outcome: you can reproduce predictions in batch scripts and the API.

In [68]:
# Save model
model_path = os.path.join(ARTIFACTS_DIR, f"model_{MODEL_TAG}.cbm")
final_model.save_model(model_path)

# Save preprocessing + channel stats references
with open(os.path.join(ARTIFACTS_DIR, f"preprocess_{MODEL_TAG}.json"), "w", encoding="utf-8") as f:
    json.dump(preprocess_full, f, ensure_ascii=False, indent=2)

# Save channel stats table (CSV is easy to inspect)
ch_stats_path = os.path.join(ARTIFACTS_DIR, f"channel_stats_{MODEL_TAG}.csv")
ch_stats_full.to_csv(ch_stats_path, index=False)

# Save channel CPM stats
ch_cpm_stats_path = os.path.join(ARTIFACTS_DIR, f"channel_cpm_stats_{MODEL_TAG}.csv")
ch_cpm_stats_full.to_csv(ch_cpm_stats_path, index=False)

with open(os.path.join(ARTIFACTS_DIR, f"meta_{MODEL_TAG}.json"), "w", encoding="utf-8") as f:
    json.dump({
        "created_at": "2026-01-10",
        "model": "CatBoostRegressor",
        "target": "log1p(VIEWS)",
        "feature_cols": feature_cols,
        "cat_features": cat_features,
        "holdout_days_for_eval": HOLDOUT_DAYS,
        "notes": "Simplified v1: dedup + channel elasticity + CTR/actions + clip/blend",
    }, f, ensure_ascii=False, indent=2)

with open(os.path.join(ARTIFACTS_DIR, f"metrics_{MODEL_TAG}_local.json"), "w", encoding="utf-8") as f:
    json.dump(metrics_local, f, ensure_ascii=False, indent=2)

print("Saved:", model_path)
print("Saved:", ch_stats_path)
print("Saved:", ch_cpm_stats_path)


Saved: artifacts/model_v1_simplified.cbm
Saved: artifacts/channel_stats_v1_simplified.csv
Saved: artifacts/channel_cpm_stats_v1_simplified.csv


## 10. Optional: fill `TestDataset.csv` and export a submission file

Outcome: `outputs/TestDataset_filled_model_<MODEL_TAG>.csv` ready for upload.


In [69]:
if os.path.exists(TESTDATA_PATH):
    test_df = pd.read_csv(TESTDATA_PATH)
    test_df.columns = test_df.columns.str.strip()
    test_df["DATE"] = pd.to_datetime(test_df["DATE"], errors="coerce")

    test_fe = apply_basic_preprocess(test_df, preprocess_full)
    test_fe = apply_channel_stats(
        test_fe,
        ch_stats_full,
        global_med_log_full,
        global_ctr_full,
        global_actions_rate_full,
        global_slope_full,
        global_cpm_log_med_full,
        global_views_clip_full,
    )
    test_fe = apply_channel_cpm_stats(test_fe, ch_cpm_stats_full, global_cpm_median_full)

    X_test = test_fe[feature_cols]
    test_pool = Pool(X_test, cat_features=cat_features)

    pred_test_raw_log = final_model.predict(test_pool)
    pred_test_raw = np.clip(np.expm1(pred_test_raw_log), 0, None)
    pred_test = post_process_predictions(
        pred_test_raw,
        test_fe,
        global_views_clip_full,
        use_clip=USE_PRED_CLIP,
        blend_alpha=BLEND_ALPHA,
        blend_base=BLEND_BASE,
        scale_factor=SCALE_FACTOR,
    )

    out = test_df.copy()
    out["VIEWS"] = np.round(pred_test).astype(int)

    out_path = os.path.join(OUTPUTS_DIR, f"TestDataset_filled_model_{MODEL_TAG}.csv")
    out.to_csv(out_path, index=False)
    print("Saved:", out_path)
else:
    print(f"Test dataset not found at: {TESTDATA_PATH}")


Saved: outputs/TestDataset_filled_model_v1_simplified.csv


## Recommended next step (Model v2)

If this simplified v1 still stalls:
- run a rolling backtest (3 windows)
- try MAE / Quantile objective
- optionally add offline TGStat/TGMaps channel features (subscribers, avg views, etc.) cached locally


In [78]:
df_answer = pd.read_csv(os.path.join(OUTPUTS_DIR, 'TestDataset_filled_model_v1_simplified.csv'))
df_answer.head(20)
df_answer['VIEWS'] = -1
df_answer.to_csv(os.path.join(OUTPUTS_DIR, 'model_to_check.csv'), index=False)


In [79]:
df_check = pd.read_csv(os.path.join(OUTPUTS_DIR, 'model_to_check.csv'))
df_check.head()


,CPM,CHANNEL_NAME,DATE,VIEWS
0,25.0,topcareerschool,2023-03-11,-1
1,10.0,topcareerschool,2023-03-11,-1
2,13.0,hrsecrets_life,2023-03-11,-1
3,7.0,hrsecrets_life,2023-03-11,-1
4,7.0,Dirclub,2023-03-12,-1


In [71]:
df_answer = pd.read_csv('/Users/karimkhabib/Documents/Projects Programming/PyCharm/telegram-ads-forecaster/outputs/TestDataset_filled.csv')
df_answer.head(20)

,CPM,CHANNEL_NAME,DATE,VIEWS
0,25.0,topcareerschool,2023-03-11,130
1,10.0,topcareerschool,2023-03-11,150
2,13.0,hrsecrets_life,2023-03-11,126
3,7.0,hrsecrets_life,2023-03-11,133
4,7.0,Dirclub,2023-03-12,150
5,18.0,equium_russia,2023-03-12,121
6,9.0,potok_ads,2023-03-12,322
7,14.0,damirkhalilov,2023-03-12,79
8,4.0,smmrus,2023-03-12,333
9,12.0,Oskar_Hartmann,2023-03-12,367
